# The Extended Kalman Filter (EKF)

*Course 3 — Nonlinear Kalman Filters, Part 1. Most real systems are nonlinear, so the linear KF's exact expectations no longer hold. The EKF is the classic fix: **linearize the model** about the current estimate and apply the [linear KF](08_Deriving_the_Linear_Kalman_Filter.ipynb) machinery. Reuses the [six SPI steps](07_Sequential_Probabilistic_Inference_Six_Steps.ipynb).*

**Style:** every equation gets a plain-language paraphrase (→); extra intuition is flagged **→ Intuition**.

### 🧩 The Nonlinear Problem

- We now have a genuinely nonlinear model:

$$
x_k = f(x_{k-1}, u_{k-1}, w_{k-1}), \qquad z_k = h(x_k, u_k, v_k).
$$

- The SPI steps still ask for expectations like $\hat{x}_k^- = \mathbb{E}[f(x_{k-1},u_{k-1},w_{k-1})\mid\mathbb{Z}_{k-1}]$.

  → For a nonlinear $f$, **the mean of $f(x)$ is *not* $f(\text{mean of }x)$**, and the output covariance is no longer a simple $A\Sigma A^T$. The clean linear formulas break.

- Three practical approximations exist: the **EKF** (linearize), the **sigma-point KF** (approximate the distribution — [13](13_Sigma_Point_Unscented_Kalman_Filter.ipynb)), and the **particle filter** (Monte-Carlo — Course 4).

- **→ Intuition:** the whole difficulty of nonlinear filtering is "how do I push a Gaussian through a curved function and still describe the result as a Gaussian?" Each method answers differently.

### 🧩 The EKF Idea — First-Order Taylor Linearization

- Approximate $f$ and $h$ by a **first-order Taylor expansion** about the current best estimate. For example, around $\hat{x}_{k-1}^{+}$:

$$
f(x_{k-1},u_{k-1},w_{k-1}) \approx f(\hat{x}_{k-1}^{+},u_{k-1},\bar{w}) + \hat{A}_{k-1}(x_{k-1}-\hat{x}_{k-1}^{+}) + \hat{B}_{k-1}(w_{k-1}-\bar{w}).
$$

  → Replace the curved function by its **tangent plane** at the current estimate. Errors and noise are assumed small enough that the linear piece dominates.

- The linearization coefficients are **Jacobian matrices** (partial derivatives evaluated at the estimate):

$$
\hat{A}_{k-1} = \left.\frac{\partial f}{\partial x}\right|_{\hat{x}_{k-1}^{+}}, \quad
\hat{B}_{k-1} = \left.\frac{\partial f}{\partial w}\right|_{\bar{w}}, \quad
\hat{C}_{k} = \left.\frac{\partial h}{\partial x}\right|_{\hat{x}_{k}^{-}}, \quad
\hat{D}_{k} = \left.\frac{\partial h}{\partial v}\right|_{\bar{v}}.
$$

  → These are the *local* $A,B,C,D$. They change every step because the tangent plane is taken at a moving point. $\hat B,\hat D$ map noise into state/output when noise enters nonlinearly.

### 🧩 The Six EKF Steps

**Prediction**
$$
\textbf{1a: } \hat{x}_k^- = f(\hat{x}_{k-1}^+, u_{k-1}, \bar{w}) \qquad \text{(propagate through the *true* nonlinear } f\text{)}
$$
$$
\textbf{1b: } \Sigma_{\tilde{x},k}^- = \hat{A}_{k-1}\Sigma_{\tilde{x},k-1}^+\hat{A}_{k-1}^T + \hat{B}_{k-1}\Sigma_{\tilde{w}}\hat{B}_{k-1}^T
$$
$$
\textbf{1c: } \hat{z}_k = h(\hat{x}_k^-, u_k, \bar{v})
$$

**Correction**
$$
\textbf{2a: } L_k = \Sigma_{\tilde{x},k}^-\hat{C}_k^T\big(\hat{C}_k\Sigma_{\tilde{x},k}^-\hat{C}_k^T + \hat{D}_k\Sigma_{\tilde{v}}\hat{D}_k^T\big)^{-1}
$$
$$
\textbf{2b: } \hat{x}_k^+ = \hat{x}_k^- + L_k\big(z_k - \hat{z}_k\big) \qquad
\textbf{2c: } \Sigma_{\tilde{x},k}^+ = (I - L_k\hat{C}_k)\,\Sigma_{\tilde{x},k}^-
$$

  → **Means (1a, 1c) use the full nonlinear functions** $f,h$ — cheap and accurate. **Covariances and the gain use the Jacobians** — the linearization only enters where we must move uncertainty around. It's the linear KF with $A,B,C,D$ replaced by local Jacobians $\hat A,\hat B,\hat C,\hat D$.

- **→ Intuition:** propagate the *point* exactly, but propagate the *uncertainty* through a linear approximation. This split is why the EKF is often "good enough" — the mean isn't crippled by linearization, only the covariance is.

### 🧩 Problems with the EKF

- **Linearization error:** if $f$ or $h$ is strongly curved over the span of the state uncertainty, the tangent-plane approximation is poor — the propagated covariance is wrong, sometimes badly, and the filter can **diverge**.

  → The EKF only captures **first-order** behavior; it ignores how curvature spreads or skews the distribution.

- **Jacobians required:** you must derive (and correctly code) analytic partial derivatives of $f$ and $h$. For messy models this is error-prone or intractable; numerical Jacobians add cost and noise.

- **Not distribution-aware:** it implicitly assumes the transformed distribution stays Gaussian and unbiased, which nonlinear maps violate.

- The **Iterated EKF (IEKF)** partly helps: re-linearize the *measurement* update repeatedly at the improved estimate $\hat x_k^{+}$ before accepting it.

  → Relinearizing at a better point tightens the correction when $h$ is very nonlinear — but it doesn't fix the prediction step and still needs Jacobians.

- **→ Intuition:** the EKF's weaknesses (Jacobians, first-order-only, divergence risk) are exactly what the **sigma-point KF** ([13](13_Sigma_Point_Unscented_Kalman_Filter.ipynb)) was designed to remove.

### 🧩 Summary

- Nonlinear models break the linear KF because $\mathbb{E}[f(x)]\neq f(\mathbb{E}[x])$ and covariance no longer propagates as $A\Sigma A^T$.

- The **EKF** linearizes $f,h$ with a **first-order Taylor** expansion, using **Jacobians** $\hat A,\hat B,\hat C,\hat D$ evaluated at the current estimate.

- It propagates **means through the exact nonlinear functions** but **covariance/gain through the Jacobians** — the same six steps as the linear KF.

- Weaknesses: linearization error (divergence risk), the burden of deriving Jacobians, and no awareness of distribution distortion; the **IEKF** mitigates the measurement step only.

---
*Next: [13 · Sigma-Point / Unscented Kalman Filter](13_Sigma_Point_Unscented_Kalman_Filter.ipynb).*